# Clinical Trial Disease Classification

## Step 1: Problem Definition
This project aims to classify clinical trial diseases based on unstructured text data (brief summaries) into predefined categories. We use NLP techniques to extract features from the text and train a classification model.

**Rationale:** We chose a Classification approach here because the goal is to automatically categorize text into discrete, predefined buckets (disease categories), which is a classic supervised learning problem.

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sqlalchemy import create_engine, Column, Integer, String, Text, Float, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker
from sqlalchemy import text
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

## Step 2: Data Collection
Connecting to SQLite and loading the raw clinical trials CSV into the database.

**Rationale for SQLite & Pandas:** SQLite is chosen because it's lightweight, highly portable, and requires no separate server setup, making it perfect for a reproducible presentation environment. Pandas is used for its highly efficient in-memory data manipulation capabilities.

In [ ]:
DATABASE_URL = "sqlite:///clinical_trials.db"
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

class ClinicalTrial(Base):
    __tablename__ = "clinical_trials"
    id = Column(Integer, primary_key=True, index=True)
    nct_id = Column(String(50), unique=True, index=True, nullable=False)
    title = Column(String(500), nullable=True)
    official_title = Column(Text, nullable=True)
    brief_summary = Column(Text, nullable=False)
    cleaned_summary = Column(Text, nullable=True)
    conditions = Column(Text, nullable=True)
    interventions = Column(Text, nullable=True)
    overall_status = Column(String(100), nullable=True)
    study_type = Column(String(100), nullable=True)
    phase = Column(String(50), nullable=True)
    sex = Column(String(50), nullable=True)
    minimum_age = Column(String(50), nullable=True)
    maximum_age = Column(String(50), nullable=True)
    healthy_volunteers = Column(String(50), nullable=True)
    eligibility_criteria = Column(Text, nullable=True)
    clinicaltrials_url = Column(String(500), nullable=True)
    disease_category = Column(String(100), index=True, nullable=False)

Base.metadata.create_all(bind=engine)

# Load data into pandas for our workflow
csv_path = r"c:\Users\jegad\projects\Clinical Trial Disease Classification\data\clinical_trials_raw_patient2trial_conditions.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df['brief_summary'] = df['brief_summary'].fillna("")
    df['title'] = df['title'].fillna(df['nct_id'])
    df['source_condition_query'] = df['source_condition_query'].fillna("other")
    df = df.rename(columns={'source_condition_query': 'disease_category'})
    print(f"Loaded {len(df)} raw records.")
else:
    print("CSV not found, proceeding with empty dataframe or existing DB.")
    df = pd.DataFrame(columns=['brief_summary', 'disease_category'])


## Step 3: Data Cleaning & Exploratory Data Analysis (EDA)
Cleaning text data (NLP) and visualizing disease distribution.

**Rationale for NLP choices:** We use NLTK's Lemmatizer instead of a Stemmer because it reduces words to their actual dictionary base form rather than just chopping off suffixes. We remove 'stop words' (like 'and', 'the') because they add noise without contributing to the core meaning of the clinical trial summaries. Median/Mode imputations are used elsewhere as they are robust to outliers.

In [ ]:
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text_fast(text_data):
    if not isinstance(text_data, str) or not text_data.strip(): return ""
    text_data = text_data.lower()
    text_data = re.sub(r'<[^>]*>', ' ', text_data)
    text_data = re.sub(r'[^a-z\s]', ' ', text_data)
    words = text_data.split()
    cleaned_words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) > 2]
    return " ".join(cleaned_words)

print("Preprocessing summaries...")
df['cleaned_summary'] = df['brief_summary'].apply(clean_text_fast)
df = df[df['cleaned_summary'].str.strip() != ""]

# --- EDA Visualizations ---
plt.figure(figsize=(12,6))
sns.countplot(data=df, y='disease_category', order=df['disease_category'].value_counts().index)
plt.title("Distribution of Disease Categories")
plt.xlabel("Count")
plt.ylabel("Disease Category")
plt.tight_layout()
plt.show()


## Step 4: Feature Engineering
Vectorizing text data using TF-IDF.

**Rationale for TF-IDF:** We selected Term Frequency-Inverse Document Frequency (TF-IDF) over simple Bag-of-Words because TF-IDF penalizes highly frequent but uninformative words across the corpus, giving more weight to terms that uniquely identify specific disease categories.

In [ ]:
X = df['cleaned_summary'].values
y = df['disease_category'].values

tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), sublinear_tf=True)
print(f"Data shapes: X: {X.shape}, y: {y.shape}")


## Step 5: Train-Test Split
Stratified splitting to maintain class proportions.

**Rationale for Stratified Split:** Medical datasets often suffer from class imbalance. Using a `stratify=y` parameter ensures that minority disease classes have the exact same proportion in both our training and testing sets, preventing skewed evaluation metrics.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set size: {len(X_train)} | Test set size: {len(X_test)}")


## Step 6: Model Selection
We use **Logistic Regression** because it typically works exceptionally well for high-dimensional, sparse TF-IDF vectors, providing a good baseline that is also highly interpretable.

**Rationale for Pipelines:** We encapsulate TF-IDF and our Classifier inside a `sklearn.pipeline.Pipeline`. This is a critical best practice to prevent 'data leakage', ensuring that vectorization statistics are fitted *only* on the training data, not the test data.

In [ ]:
clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
pipeline = Pipeline([
    ('tfidf', tfidf),
    ('clf', clf)
])


## Step 7: Model Training
Fitting the Logistic Regression model on the training set.

In [ ]:
print("Training model (TF-IDF + Logistic Regression)...")
pipeline.fit(X_train, y_train)
print("Model training complete!")


## Step 8: Model Evaluation
Evaluating accuracy and visualizing the confusion matrix.

**Rationale for Confusion Matrix:** While overall Accuracy is a good high-level metric, a Confusion Matrix allows us to visually inspect exactly where the model is misclassifying diseases (False Positives vs. False Negatives), which is crucial in clinical contexts.

In [ ]:
y_pred = pipeline.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=pipeline.classes_, yticklabels=pipeline.classes_)
plt.title("Confusion Matrix")
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.xticks(rotation=45)
plt.show()


## Step 9: Model Tuning
Using `GridSearchCV` to find the optimal hyperparameters.

**Rationale for GridSearchCV:** Instead of manually guessing the best model parameters, GridSearchCV systematically searches through a grid of possible parameters (like the regularization strength `C`). It uses Cross-Validation internally to ensure the tuned model generalizes well and doesn't just overfit the training data.

In [ ]:
# Defining parameter grid for GridSearchCV
param_grid = {
    'clf__C': [0.1, 1.0, 1.5, 2.0]
}

print("Starting Grid Search...")
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")

# Re-evaluate with best estimator
best_pipeline = grid_search.best_estimator_
y_pred_tuned = best_pipeline.predict(X_test)
print(f"Tuned Model Accuracy on Test Set: {accuracy_score(y_test, y_pred_tuned):.4f}")


## Step 10: Model Deployment
Saving the best model to disk for deployment.

**Rationale for Joblib:** We use `joblib.dump` over standard python `pickle` because `joblib` is heavily optimized for saving objects that contain large numpy arrays (which is true for fitted TF-IDF matrices and scikit-learn models), resulting in faster saves and smaller file sizes.

In [ ]:
models_dir = "models"
os.makedirs(models_dir, exist_ok=True)
model_path = os.path.join(models_dir, "classifier_pipeline_best.joblib")

joblib.dump(best_pipeline, model_path)
print(f"SUCCESS: Tuned model pipeline saved successfully to {model_path}!")
